#  TESTING  LDM COMPONENTS

In [ ]:
from transformers import CLIPTokenizer,CLIPTextModel
from diffusers import AutoencoderKL
import torch
import torch.nn  as nn
  
class Clip_VAE(nn.Module):
    def __init__(self,
                model_name:  str,
                device,
                tokenizer = None,
                text_encoder = None,
                Vae = None 
                 ):
        """Module for clip tokenizer,text_encoder and vae decoder/encoder

        Args:
            text (str): specify component type(CLIP / VAE)
            device (cuda/cpu): device
            tokenizer : CLIPTokenizer, for input_ids or tokens
            text_encoder : CLIPTextModel, to  encode ids or  tokens to embeddings
            Vae : AutoencoderKL
        """
        super().__init__()
        self.model_name =  model_name
        self.device = device
        self.tokenizer  = tokenizer
        self.text_encoder = text_encoder
        self.Vae = Vae
        
        if self.model_name =='clip':
            self.get_tokens = self.tokenizer.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='tokenizer')
            self.get_embeddings = self.text_encoder.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='text_encoder').to(self.device)
            
        if  self.model_name == 'Vae_encode' or self.model_name == 'Vae_decode':
            self.latent_scaling_factor =  0.18215
            self.autoencoder = self.Vae.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='vae').to(device)
            
    def forward(self,input : str | torch.Tensor)-> torch.Tensor:
        if self.model_name=='clip':
            input_ids = self.get_tokens([input],
                                       padding='max_length',
                                       max_length = self.get_tokens.model_max_length,
                                       truncation=True, 
                                       return_tensors="pt",
                                       )['input_ids'] 
            input_ids_tensor =  input_ids.to(self.device)  
            with torch.no_grad():
                text_embeddings =   self.get_embeddings(input_ids_tensor)['last_hidden_state']
            return  text_embeddings,text_embeddings.shape
        input = input.to(self.device)
        if  self.model_name == 'Vae_encode':
            with torch.no_grad():
                encoded = self.latent_scaling_factor * self.autoencoder.encode(input).sample()
            return encoded
        if  self.model_name == 'Vae_decode':
            with torch.no_grad():
                decoded  = self.autoencoder.decode(input/self.latent_scaling_factor).sample()
                
            return (decoded/2 + 0.5).clamp(0,1) # [-1,1] to [0,1]
            

: 

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

: 

: 

In [ ]:
text = 'i am a boy'
encode_text = Clip_VAE('clip',device,CLIPTokenizer,CLIPTextModel)
encode_text(text)

(tensor([[[-0.3884,  0.0229, -0.0522,  ..., -0.4899, -0.3066,  0.0675],
          [-1.2454, -0.3497,  1.3990,  ..., -1.4180,  0.4403,  0.1712],
          [ 0.2763, -0.1045,  1.2606,  ..., -0.1392, -0.0854, -0.0184],
          ...,
          [ 1.2518, -0.5856, -0.4464,  ..., -0.2005,  0.6684,  0.0709],
          [ 1.2504, -0.5913, -0.4326,  ..., -0.1654,  0.6799,  0.0588],
          [ 1.2016, -0.5140, -0.3981,  ..., -0.1803,  0.6750,  0.0074]]],
        device='cuda:0'),
 torch.Size([1, 77, 768]))

: 

: 

In [4]:
## UNEt
import torch
import torch.nn  as nn
class  TimeEmbedding(nn.Module):
    """Encodes scalar diffusion steps into continuous vectors.
    """
    def  __init__(self,base_dim,f_dim):
        super().__init__()
        self.base_dim = base_dim # or d_model
        self.f_dim=  f_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(self.base_dim,self.f_dim),
            nn.SiLU(),
            nn.Linear(self.f_dim,self.f_dim)
        )
    def forward(self,timesteps: torch.Tensor)-> torch.Tensor:
        time_vector = timesteps.flatten().float()#(len(steps),)
        dim_vector = torch.arange(0,self.base_dim,2).float()#(base_dim/2,)
        frequency_scales = 10000 ** (dim_vector/self.base_dim)
        f_s =  1/frequency_scales
        #final = time_vector[:,None] @ f_s[None,:]  # (steps,1) x (1,base_dim/2)
        final = torch.outer(time_vector,f_s)#(steps,base_dim/2)
        embedding = torch.zeros(len(timesteps),self.base_dim) #(steps,base_dim)
        embedding[:,0::2] = torch.sin(final)
        embedding[:,1::2]  = torch.cos(final)
        context =  self.mlp(embedding)  #(dim or d_model,final dim)
        return context.shape

In [5]:
t  = torch.arange(0,1000,1)
embed =  TimeEmbedding(320,1280)
embed(t)

torch.Size([1000, 1280])

In [36]:
def Normalize(in_channels, num_groups=32):
    return torch.nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, eps=1e-6, affine=True)

class ResnetBlock(nn.Module):
    """Processes spatial image features and injects time context.

    Attributes:
        in_ch : input channel dim
        out_ch : output channel dim
    """
    def __init__(self,in_ch,d_t_embed,out_ch,dropout):
        super().__init__()
        self.embed_dim = d_t_embed
        self.in_channels = in_ch
        self.out_channels = out_ch  
        self.inps = nn.Sequential(
                Normalize(in_channels=self.in_channels),
                nn.SiLU(),
                nn.Conv2d(
                        self.in_channels,
                        self.out_channels,
                        kernel_size=3,
                        padding=1,
                    )
        )
        
        # Project time embeddings
        self.time_proj = nn.Linear(self.embed_dim,self.out_channels)
        self.outs = nn.Sequential(
                Normalize(in_channels=self.out_channels),
                nn.SiLU(),
                nn.Dropout(0.1),
                nn.Conv2d(
                        self.out_channels,
                        self.out_channels,
                        kernel_size=3,
                        padding=1,
                    )
        )
        
        if self.in_channels == self.out_channels:
            self.skip_connection = nn.Identity()
        else:
            self.skip_connection = nn.Conv2d(
                                    self.in_channels,
                                    self.out_channels, 
                                    1
                                    )    
    def forward(self,x,time_context):
        """
            (x)Image feats: (Batch, In_Channels, H, W)
            Time context: (Batch, 1280)
        """
        h = x
        h =  self.inps(x)
        # project time context
        proj  = self.time_proj(time_context)[:,:,None,None] #(b,feats,1,1)
        h =  h + proj
        h =  self.outs(h)
        return self.skip_connection(x) + h
        

 

In [37]:
block = ResnetBlock(in_ch=320,d_t_embed=1280, out_ch=640,dropout=0.1)
mock_latents = torch.randn(2, 320, 64, 64)       # Shape: [Batch, Channels, H, W]
mock_time_emb = torch.randn(2, 1280)             # Shape: [Batch, Time_Dim]
output = block(mock_latents, mock_time_emb)
print("Output Shape:", output.shape)


Output Shape: torch.Size([2, 640, 64, 64])


In [79]:
import torch.nn.functional as F

class CrossAttn(nn.Module):
    def __init__(self,context_dim,channel_dim):
        super().__init__()
        self.context_dim = context_dim
        self.q = nn.Linear(channel_dim,channel_dim,bias=False)
        self.k = nn.Linear(context_dim,channel_dim,bias=False)
        self.v = nn.Linear(context_dim,channel_dim,bias=False)
        self.out = nn.Linear(channel_dim,channel_dim)
    def forward(self,x,context_matrix):
        '''
        Image sequence(x) : [b,c,h,w]
         context  matrix : [b,seq,context_dim]
        '''
        b,c,h,w = x.size()
        x = x.permute(0,2,3,1).view(b,h*w,c)
        q  = self.q(x) #(b,seq_q,c)
        k = self.k(context_matrix)#(b,seq_k,c) 
        v = self.v(context_matrix)#(b,seq_v,c)
        scores = q @ k.transpose(-2,-1)#(b,seq_q,seq_k)
        scale = q.shape[-1] ** 0.5
        scaled = scores / scale
        prob = F.softmax(scaled,dim=-1)
        result = prob @ v #(b,seq_q,seq_k)
        return self.out(result) #(b,seq,c)
        

In [80]:
img = torch.randn(1,640,64,64)
text = torch.randn(1,77,768)
c = CrossAttn(768,640)
c(img,text).shape

torch.Size([1, 4096, 640])

In [78]:
b,c,h,w = img.size()
print(h)

64


In [ ]:
class UpSample(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.out =  nn.Upsample(scale_factor=2)
        self.conv = nn.Conv2d(channels,channels,3,padding=1)
    def forward(self,x):
        res  = self.out(x)
        return self.conv(res)    
   
class DownSample(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.conv = nn.Conv2d(channels,channels,3,stride=2,padding=1)
    def forward(self,x):
        return self.conv(x)    
     
    